# ZollerQu: long/short stock split (Qollab × IonQ hackathon)

This effort is supported by Qollab & IonQ.

Working notebook, built cell by cell. Local development only: the Qollab entry point will be a separate `main.py`, and the reusable code lives in the `stocks/` package next to this notebook.

## Task 1: Load and validate the OR-Library dataset

Source: J E Beasley, OR-Library, `indtrack1.txt` (index tracking problem 1, MIT licence). Downloaded once during development; nothing is downloaded at run time.

In [1]:
import hashlib
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "stocks").is_dir() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from stocks import data, validate_data

print("repo root:", ROOT)

repo root: /Users/t/zollerqu-qollab


### 1.1 Source file as downloaded

In [2]:
raw = data.RAW_PATH
lines = raw.read_text().splitlines()
print("file:  ", raw.relative_to(ROOT), f"({raw.stat().st_size} bytes)")
print("sha256:", hashlib.sha256(raw.read_bytes()).hexdigest())
print("line 1:", repr(lines[0]))
print("line 2:", repr(lines[1]))
values_per_line = [len(line.split()) for line in lines[1:]]
print("lines after header:", len(values_per_line),
      "| with 4 values:", values_per_line.count(4),
      "| with 3 values:", values_per_line.count(3),
      "| total values:", sum(values_per_line))

file:   data/raw/indtrack1.txt (116665 bytes)
sha256: 258b917e304e21c87ae481b9baba0aefe877b657fa63483255c23162a74aa629
line 1: ' 31 290 '
line 2: ' 8749.31759356 8713.53262793 9001.60515134 8903.30299873 '
lines after header: 2336 | with 4 values: 2304 | with 3 values: 32 | total values: 9312


### 1.2 Parse and validate

`parse_orlib` accepts a layout only if the price count matches it **and** every series starts on a new line. It also checks that the first index price is near 8749 and that all prices are finite and positive. Any problem raises `DataFormatError`; nothing is dropped or filled in.

In [3]:
market, report = data.parse_orlib(raw)
print(report)
print("index:", market.index.shape, "| stocks:", market.prices.shape, "| names:", market.names[0], "...", market.names[-1])
print("first index price:", float(market.index[0]))
print(f"index range {market.index.min():.2f} .. {market.index.max():.2f} | "
      f"stock range {market.prices.min():.4f} .. {market.prices.max():.4f}")

ParseReport(header=(31, 290), n_values=9312, layout='(N+1)*(T+1)', n_series=32, observations=291)
index: (291,) | stocks: (291, 31) | names: s01 ... s31
first index price: 8749.31759356
index range 8713.53 .. 29550.18 | stock range 1.7928 .. 484.5881


### 1.3 Normalised CSV `data/hangseng_weekly.csv`

One row per observation: `week,index,s01,...,s31`. The source has no dates, so `week` is the observation number 0..290 in source order.

In [4]:
data.write_csv(market, data.CSV_PATH)
reloaded = data.load_csv(data.CSV_PATH)
same = (reloaded.names == market.names
        and (reloaded.index == market.index).all()
        and (reloaded.prices == market.prices).all())
print("reloaded identical to the parsed source:", same)
for line in data.CSV_PATH.read_text().splitlines()[:3]:
    print(line[:100] + " ...")

reloaded identical to the parsed source: True
week,index,s01,s02,s03,s04,s05,s06,s07,s08,s09,s10,s11,s12,s13,s14,s15,s16,s17,s18,s19,s20,s21,s22,s ...
0,8749.31759356,9.33675195,14.37788798,7.36644878,29.46579511,5.55589992,8.55573087,16.41919306,9.95 ...
1,8713.53262793,9.86926631,14.18263271,7.40194974,27.33573763,5.44939705,8.50247943,16.41919306,9.28 ...


### 1.4 Licence notice

In [5]:
ok, message = validate_data.check_licence()
print(message)
print()
print("\n".join(data.LICENSE_PATH.read_text().splitlines()[:10]))

verified (data/LICENSE_ORLIB.txt holds the full MIT notice, copied verbatim from OR-Library legal.html)

OR-Library licence notice for data/raw/indtrack1.txt and data/hangseng_weekly.csv

Data:    indtrack1.txt (index tracking test problem 1), OR-Library, J E Beasley
         https://people.brunel.ac.uk/~mastjjb/jeb/orlib/files/indtrack1.txt
Notice:  copied verbatim from https://people.brunel.ac.uk/~mastjjb/jeb/orlib/legal.html
         on 2026-10-10. Only the HTML markup was removed; the wording, line breaks and
         the original "(J E Beasley}" bracket are unchanged.
Changes: data/hangseng_weekly.csv holds the same prices rearranged into columns by
         stocks/data.py (team ZollerQu). No values were added, removed or altered.


### 1.5 Task 1 report (also runs the 23 unit tests)

Same output as `python -m stocks.validate_data` from the repo root.

In [6]:
exit_code = validate_data.main()

[TASK 1] OR-Library dataset validation
- Source: original OR-Library file data/raw/indtrack1.txt (sha256 258b917e304e21c8...)
- Header: 31 290
- Layout: (N+1)*(T+1) = 32 series x 291 = 9312 prices, each series starting on a new line
- Stock series: 31
- Index series: 1
- Price observations per series: 291 (index: 291)
- First index price: 8749.31759356
- Invalid or missing prices: 0 (counted in the written CSV)
- CSV output: data/hangseng_weekly.csv (291 rows x 33 columns, reload identical to source: yes)
- Licence notice: verified (data/LICENSE_ORLIB.txt holds the full MIT notice, copied verbatim from OR-Library legal.html)
- Tests: 23 passed, 0 failed
RESULT: PASS
